**2D**
1) Define the domain $\Omega$
2) Generate triangulations T_h
3) Construct finite element space V_h
4) Get local stiffness matrix A_k
5) Get local mass matrix M_k
6) Assemble local stiffness matrices into global stiffness matrix A
7) Assemble local mass matrices into a global mass matrix M
8) Impose Dirichlet Boundary Condition
9) Solve $A * u = \lambda * M * u$
10) Extract eigenvalues
11) compute the eigenvectors
12) Compare with exact solution


**2D FEM Implementation notes**
1. Generated mesh using Delaunay triangulation
2. Computed local element matrices
3. Assembled global stiffness and mass matrices
4. Identified boundaru and interior nodes
5. Imposed homogeneous Dirichlet boundary conditions by removing rows and columns associated with boundary nodes
6. Solved reduced generalised eigenvalue problem

*For future*
- global matrices are sparse (majority of elements are 0) and symmetric
- boundary conditions reduce the dimention of the system
- enumerate shows which node number corresponds to the coordinates
- with 1 interior node we have 1 degree of freedom, the eigenvalue was 48, comparing to the exact first Dirichlet eigenvalue on the unit square is $\lambda_1 = 2 * \pi^2 \approx 19.739$
- the exact eigenvalue for the unit square with Dirichlet BC is $\lambda_m,n = \pi^2 (m^2 + n^2)$
- with a finer mesh, where there are 5 points on the axis the eigenvalue was 22.506 which is much closer to the real one 19.739
- for generalised FEM eigenproblem the common normalisation is $u^T M u = 1$
- shape is more important when printing the eigenvector (it is 0 on the boundary and maximum in the centre), which is the discrete approximation of the first eigenfunction of the unit square $u(x, y) = sin(\pi x) * sin(\pi y)$. If we take the finer mesh the numbers will be the coefficients of the $u_h = \sum_i u_i \phi_i (x)$. 

**The vector tells us how much of each basis function is present in the final FEM approximation**.
*The numbers in the eigenvector are coefficients at the interior nodes, not values at every point of the domain.*

In [ ]:
# 2D let the domain be uniform and defined on [0, 1]x[0, 1]
import numpy as np
from scipy.linalg import eigh
from scipy.spatial import Delaunay
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-poster")

In [ ]:
# generate mesh
def generate_mesh(n_points):
       # define the axis intervals
       x = np.linspace(0, 1, n_points)
       y = np.linspace(0, 1, n_points)

       # coordinate generation via meshgrid (takes 1D arrays and duplicates them to build 2D grids)
       X, Y = np.meshgrid(x, y)

       # c_ matches the first X with the first Y, second X with second Y etc
       domain = np.c_[X.ravel(), Y.ravel()]
       # ravel() takes 2D matrix structure and reads it row by row into a long single list of coordinates
       # Delaunay cannot read 2D grid, thats why we flatten X and Y, so they can be paired together

       # create triangles on the domain
       tri = Delaunay(domain)

       return domain, tri

# method for putting values on the main diagonal
def put_value_in_special_index(matrix, grad_phi_i, grad_phi_j, index):
       # find the dot product of 2 gradients
       grad_phi_i_j = np.dot(grad_phi_i, grad_phi_j)
       
       matrix.put(index, grad_phi_i_j)

       return matrix

# stiffness matrix
def stiffness_matrix_A(grad_phi):
       # create a matrix (interior_nodes x interior_nodes) of zeros
       A_lower_tri = np.zeros((3, 3), dtype=float)
       # off diagonal values
       # first make lower triangular matrix
       # A_2_1
       put_value_in_special_index(A_lower_tri, grad_phi[1], grad_phi[0], 3)
       # A_3_1
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[0], 6)
       # A_3_2
       put_value_in_special_index(A_lower_tri, grad_phi[2], grad_phi[1], 7)

       A = create_symmetric_matrix(A_lower_tri)
       # diagonal values
       put_value_in_special_index(A, grad_phi[0], grad_phi[0], 0)
       put_value_in_special_index(A, grad_phi[1], grad_phi[1], 4)
       put_value_in_special_index(A, grad_phi[2], grad_phi[2], 8)
       return A

def create_symmetric_matrix(lower_tri_matrix):
       # transpose the lower triangular matrix
       lower_tri_matrix_T = lower_tri_matrix.T
       # create symmetric matrix by adding the lower triangular matrix to its transpose
       sym_matrix = lower_tri_matrix + lower_tri_matrix_T

       return sym_matrix

# mass matrix
def M_loc():
       
       return np.array(
              [
                     [2, 1, 1],
                     [1, 2, 1],
                     [1, 1, 2]
              ]
       )


# find the stiffness and mass matrices for individual triangle
def triangle_solver(coords_of_triangle):

       # find the area of the triangle
       col = np.array([1, 1, 1])
       # create 3x3 matrix to find the area
       coords_matrix = np.hstack((coords_of_triangle, np.atleast_2d(col).T))

       # area of a triangle
       T_k = 0.5 * abs(np.linalg.det(coords_matrix))

       x = []
       y = []
       # for coordinate in all of the coordinates of the nodes of this triangle
       for coord in coords_of_triangle:
              x.append(float(coord[0]))
              y.append(float(coord[1]))

       c = []   
       c.append(x[2] - x[1])
       c.append(x[0] - x[2])
       c.append(x[1] - x[0])

       b = []
       b.append(y[1] - y[2])
       b.append(y[2] - y[0])
       b.append(y[0] - y[1])

       # find the gradients of the basis functions
       grad_phi = np.array([
              (1 / (2 * T_k)) * np.array([b_i, c_i]) for b_i, c_i in zip(b, c)
       ])


       # get local stiffness matrix
       A_local = T_k * stiffness_matrix_A(grad_phi)
       
       # get local stiffness matrix
       M_local = (T_k / 12) * M_loc()
       
       return A_local, M_local

# put the triangle in the global matrix
def put_local_to_global(global_matrix, local_matrix, coord):

       n_local = local_matrix.shape[0]

       for a in range(n_local):
              for b in range(n_local):
                     global_matrix[coord[a], coord[b]] += local_matrix[a, b]
       

       return global_matrix

# get boundary and interior nodes
def get_boundary_and_interior_nodes(domain):
       boundary_nodes = []
       interior_nodes = []

       for i, (x, y) in enumerate(domain):
              # check if any of the coordinates are on the boundary
              if x == 0 or x == 1 or y == 0 or y == 1:
                     boundary_nodes.append(i)
              else:
                     interior_nodes.append(i)

       return boundary_nodes, interior_nodes

# get global matrices
def get_global_matrices(coords_of_tri, domain, n_nodes):

       A_global = np.zeros((n_nodes, n_nodes), dtype=float)
       M_global = np.zeros((n_nodes, n_nodes), dtype=float)

       # for every trinagle in the mesh
       for triangle in tri_coord_sort:
              coords = domain[triangle]

              A_local, M_local = triangle_solver(coords)
              global_coords = triangle.tolist()

              put_local_to_global(A_global, A_local, global_coords)
              put_local_to_global(M_global, M_local, global_coords)

       return A_global, M_global

def apply_dirichlet(A_global, M_global, interior_nodes):
       # reducing matrices based on boundary condition, that u = 0 on the boundary
       A_reduced = A_global[np.ix_(interior_nodes, interior_nodes)]
       M_reduced = M_global[np.ix_(interior_nodes, interior_nodes)]

       return A_reduced, M_reduced

def first_eigval_error(comp_eigval):
       real_eigval = 2 * (np.pi)**2
       error = abs(real_eigval - comp_eigval)
       return error

def get_table(headers, indexes, table):
       
       df = pd.DataFrame(table, columns = headers, index = indexes)
       return df


In [ ]:
# visualisations
# visualise the mesh

def visualise_mesh(domain, triangle):
       plt.triplot(domain[:,0], domain[:,1], triangle.simplices.copy())
       plt.plot(domain[:,0], domain[:,1], "o")

       # to see the nodes on the graph
       # enumerate shows which node number corresponds to the coordinates
       for i, (x, y) in enumerate(domain):
              plt.text(x, y, f"P{i}", fontsize=12)

       # labeling the triangles
       for k, tri in enumerate(triangle.simplices):

              centroid = domain[tri].mean(axis=0)

              plt.text(centroid[0], centroid[1], f"T{k}", color="red")

       plt.gca().set_title("Mesh visualisation")
       # gca - get current axes
       # set_aspect("equal") prevents stretching the plot, if it's square it will look like square
       plt.gca().set_aspect("equal")
       plt.show()

# visualise the FEM function reconstructed from the nodal values
def visualise_FEM(eigenvectors, domain, tri, n_nodes, interior_nodes, interior_dim):
       for k in range(eigenvectors.shape[1]):
              
              v_k = eigenvectors[:, k]

              u = np.zeros(n_nodes)
              u[interior_nodes] = v_k

              plt.tripcolor(
                     domain[:, 0],
                     domain[:, 1],
                     tri.simplices,
                     u,
                     shading="gouraud" #flat
              )

              plt.colorbar()
              plt.gca().set_aspect("equal")
              plt.gca().set_title(f"Eigenfunction with {k+1}-th eigenvector")
              
              #plt.gca().legend(title=f"v_{k+1} = \n {v_k.reshape(interior_dim, interior_dim)}", loc="upper left")
              
              plt.show()

              if k == 2:
                     break
              else:
                     continue



def visualise_convergence(dof, errors):
       fig, ax = plt.subplots(figsize=(10, 8))
       ax.plot(dof, errors, marker="o")
       ax.set_xlabel(r"Degrees of Freedom")
       ax.set_ylabel(r"$E(h) = |\lambda_1 - \lambda_1^h|$")
       ax.set_title("Convergence of the FEM eigenvalues")
       ax.grid(True)
       plt.show()
       

In [ ]:
# sanity checks
def sanity_check(A_global, M_global):
       print("\nCHECKS FOR MATRIX SYMMETRY AND ROW SUM CHECK\n")
       print(f"Is the global stiffness matrix symmetric? {np.allclose(A_global, A_global.T)}")
       print(f"Is the global mass matrix symmetric? {np.allclose(M_global, M_global.T)}")

       # may have floating point error 
       print(f"Is the row sum of the global stiffness matrix is 0? {np.allclose(A_global.sum(axis=1), 0)}")


In [ ]:
# More complex mesh 
#######
##### MAIN #####
########

n_points_list = [3, 5, 9, 17]
rows = []
indexes = []
index = 1

errors = []
dofs = []

for n_points in n_points_list:

       row = []
       row.append(n_points)
       # create mesh
       domain, triangle = generate_mesh(n_points)
       

       tri_coord_sort = np.sort(triangle.simplices)


       visualise_mesh(domain, triangle)

       # get global matrices
       n_nodes = len(domain)

       A_global, M_global = get_global_matrices(tri_coord_sort, domain, n_nodes)

       # check if the matrices are symmetric and if the row sum is 0
       sanity_check(A_global, M_global)

       # get boundary and interior nodes
       boundary_nodes, interior_nodes = get_boundary_and_interior_nodes(domain)
       row.append(len(interior_nodes))
       dofs.append(len(interior_nodes))
       # how many elements we have on the side of the interior square
       interior_dim = int(np.sqrt(len(interior_nodes)))

       # apply dirichlet BC
       A_reduced, M_reduced = apply_dirichlet(A_global, M_global, interior_nodes)

       # finding eigenvalues
       eigvals, eigvecs = eigh(A_reduced, M_reduced)
       row.append(eigvals[0])
       # SciPy stores eigvectors as columns
       print(f"The eigenvalues: \n{eigvals[0]}\n")

       error = first_eigval_error(eigvals[0])
       row.append(error)
       errors.append(error)

       rows.append(row)
       indexes.append(index)
       index += 1

       visualise_FEM(eigvecs, domain, triangle, n_nodes, interior_nodes, interior_dim)

headers = ["Points per axis", "Interior DoF", "First eigenvalue", "Error"]
table = get_table(headers, indexes, rows)
print(table)

visualise_convergence(dofs, errors)
       

